In [ ]:
import pandas as pd
import requests
import json
import matplotlib.pyplot as plt


In [ ]:
import matplotlib.patches as mpatches
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [ ]:
print("Libraries imported successfully.")
print(f"Pandas version: {pd.__version__}")
print(f"Requests version: {requests.__version__}")
print(f"load at  : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Libraries imported successfully.
Pandas version: 2.2.2
Requests version: 2.32.4
load at  : 2026-05-27 09:15:26


In [ ]:
raw_df=pd.read_csv('messy_sales_data.csv')
print(f"rows  : {raw_df.shape[0]}")
print(f"columns : {raw_df.shape[1]}")
print(f"columns : {raw_df.columns.tolist()}")

rows  : 30
columns : 9
columns : ['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'order_date', 'city', 'sales_rep']


In [ ]:
raw_df=pd.read_csv('messy_sales_data.csv')
print(f"rows  : {raw_df.shape[0]}")
print(f"columns : {raw_df.shape[1]}")
print(f"columns : {raw_df.columns.tolist()}")
print("\n first 5 rows")
raw_df.head()

rows  : 30
columns : 9
columns : ['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'order_date', 'city', 'sales_rep']

 first 5 rows


,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep
0,1001,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma
1,1002,Priya Nair,NaN,Electronics,1.0,15000,2024-01-07,Delhi,Sunita Rao
2,1003,AMIT VERMA,Keyboard,Accessories,3.0,1200,2024-01-08,Bangalore,Anil Sharma
3,1004,Sunita Patel,Monitor,Electronics,NaN,22000,2024-01-10,Chennai,Ravi Kumar
4,1005,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma


In [ ]:
print("\n missing values per colummn:")
print(raw_df.isnull().sum())
print(f"\n[2] duplicate row:{raw_df.duplicated().sum()}")
print("\n[3] data types:")
print(raw_df.dtypes)
print("\n[4] unique category:",raw_df['category'].dropna().unique().tolist())
print("[4] sample order dates:",raw_df['order_date'].unique()[:8].tolist())
print("[4] sample names:",raw_df['customer_name'].dropna().unique()[:8].tolist())



 missing values per colummn:
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64

[2] duplicate row:0

[3] data types:
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object

[4] unique category: ['Electronics', 'Accessories']
[4] sample order dates: ['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '07-01-2024', '2024-01-12', '2024-01-13', '2024-01-15']
[4] sample names: ['Ramesh Kumar', 'Priya Nair', 'AMIT VERMA', 'Sunita Patel', 'kiran mehta', 'Deepak Singh', 'Ananya Das', 'Vikram Iyer']


In [ ]:
df=raw_df
print(f"working copy created with shape:{df.shape}")


working copy created with shape:(30, 9)


In [ ]:
from numpy import median
print(f"Total missing value BEFORE fix :{df.isnull().sum().sum()}")
df['customer_name'].fillna('unknown customer',inplace=True)
median_qty=df['quantity'].median()
df['quantity'].fillna(median_qty,inplace=True)
print(f"Total missing value AFTER fix :{df.isnull().sum().sum()}")
df['category'].fillna('uncategorized',inplace=True)
print(f"unique category:{df['category'].dropna().unique().tolist()}")



Total missing value BEFORE fix :7
Total missing value AFTER fix :2
unique category:['Electronics', 'Accessories', 'uncategorized']


In [ ]:
print(f"\n[2] duplicate row:{df.duplicated().sum()}")



[2] duplicate row:0


In [ ]:
print(f"row before deduplication:{len(df)}")
print(f"duplicate row found:{df.duplicated().sum()}")
duplicate_mask=df.duplicated(keep=False)
print("\n the duplicate rows (all copies shown)")
print(df[duplicate_mask][['order_id','customer_name','product','order_date']].to_string(index=False))
df.drop_duplicates(inplace=True)
df.reset_index(drop=True,inplace=True)
print(f"row after deduplication:{len(df)}")
print(f"row removed :{len(raw_df)-len(df)}")

row before deduplication:30
duplicate row found:0

 the duplicate rows (all copies shown)
Empty DataFrame
Columns: [order_id, customer_name, product, order_date]
Index: []
row after deduplication:30
row removed :0


In [ ]:
df['order_date']=pd.to_datetime(df['order_date'],dayfirst=False,errors='coerce')
nat_mask=df['order_date'].isnull()
df.loc[nat_mask,'order_date']=pd.to_datetime(df.loc[nat_mask,'order_date'],dayfirst=True,errors='coerce')
nat_count=df['order_date'].isnull().sum()
print(f"\nunparseable dates remaining (NaT):{nat_count}")
df['year']=df['order_date'].dt.year
df['month']=df['order_date'].dt.month
df['month_name']=df['order_date'].dt.strftime('%B')
df['day_name']=df['order_date'].dt.strftime('%A')
print("\nsample date after parsing:")
print(df[['order_date','year','month','month_name','day_name']].head(8).to_string(index=False))


unparseable dates remaining (NaT):2

sample date after parsing:
order_date   year  month month_name  day_name
2024-01-05 2024.0    1.0    January    Friday
2024-01-07 2024.0    1.0    January    Sunday
2024-01-08 2024.0    1.0    January    Monday
2024-01-10 2024.0    1.0    January Wednesday
2024-01-05 2024.0    1.0    January    Friday
       NaT    NaN    NaN        NaN       NaN
2024-01-12 2024.0    1.0    January    Friday
2024-01-13 2024.0    1.0    January  Saturday


In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'],dayfirst=False,errors='coerce')
nat_mask=df['order_date'].isnull()
df.loc[nat_mask,'order_date']=pd.to_datetime(df.loc[nat_mask,'order_date'],dayfirst=True,errors='coerce')

In [ ]:
print("name before standardization (sample):")
print(df['customer_name'].unique()[:8].tolist())
df['customer_name']=df['customer_name'].str.title()
print("\nname after standardization (sample):")
print(df['customer_name'].unique()[:8].tolist())

name before standardization (sample):
['Ramesh Kumar', 'Priya Nair', 'AMIT VERMA', 'Sunita Patel', 'kiran mehta', 'Deepak Singh', 'unknown customer', 'Ananya Das']

name after standardization (sample):
['Ramesh Kumar', 'Priya Nair', 'Amit Verma', 'Sunita Patel', 'Kiran Mehta', 'Deepak Singh', 'Unknown Customer', 'Ananya Das']


In [ ]:
wrong_mask=(df['product']=='keyboard') & (df['category']=='electronics')
print(f"row ro fix:{wrong_mask.sum()}")
print("before fix:")
print(df[wrong_mask][['order_id','product','category']].to_string(index=False))
df.loc[wrong_mask,'category']='computer accessories'
print("\nafter fix:")
print(df[wrong_mask][['order_id','product','category']].to_string(index=False))
print("\n all unique category:")
print(df['category'].unique().tolist())


row ro fix:0
before fix:
Empty DataFrame
Columns: [order_id, product, category]
Index: []

after fix:
Empty DataFrame
Columns: [order_id, product, category]
Index: []

 all unique category:
['Electronics', 'Accessories', 'uncategorized']


In [ ]:
df['quantity'] =pd.to_numeric(df['quantity'],errors='coerce').astype(int)
df['unit_price']=pd.to_numeric(df['unit_price'],errors='coerce')
df['revenue']=df['quantity']*df['unit_price']
print("revenue column created successfully")
print("\nsample :quantity x unit_price =revenue")
print(df[['customer_name','product','quantity','unit_price','revenue']].head(8).to_string(index=False))



revenue column created successfully

sample :quantity x unit_price =revenue
   customer_name    product  quantity  unit_price  revenue
    Ramesh Kumar     Laptop         2       45000    90000
      Priya Nair        NaN         1       15000    15000
      Amit Verma   Keyboard         3        1200     3600
    Sunita Patel    Monitor         2       22000    44000
    Ramesh Kumar     Laptop         2       45000    90000
     Kiran Mehta      Mouse        10         800     8000
    Deepak Singh Headphones         2        3500     7000
Unknown Customer     Webcam         1        2500     2500


In [ ]:
output_filename = 'clean_sales_data.csv'
df.to_csv(output_filename, index=False)

print(f"Cleaned data saved to: {output_filename}")
print(f"Final dataset: {df.shape[0]} rows x {df.shape[1]} columns")
print("ETL Pipeline for Sales Data: COMPLETE")
print("EXTRACT messy_sales_data.csv loaded")
print("TRANSFORM + 6 issues fixed (nulls, dupes, dates, names, categories, types)")
print(f"LOAD {output_filename} saved")


Cleaned data saved to: clean_sales_data.csv
Final dataset: 30 rows x 14 columns
ETL Pipeline for Sales Data: COMPLETE
EXTRACT messy_sales_data.csv loaded
TRANSFORM + 6 issues fixed (nulls, dupes, dates, names, categories, types)
LOAD clean_sales_data.csv saved


In [ ]:
from re import search
SERP_API_KEY='e94af6c8514fa35bad847c9ba4c82193e015ad2f39a29d524560b986c0b3f365'
SERP_URL='https://serpapi.com/search.json'
SEARCH_QUERY='DATA ENGINEERING'
print(f"SerpAPI Key  : {'Set (live data)' if SERP_API_KEY != 'YOUR_SERPAPI_KEY_HERE' else 'Not set (fallback data will be used)'}")
print(f"SEarch query :{search}")

SerpAPI Key  : Set (live data)
SEarch query :<function search at 0x7bccd2d477e0>


In [ ]:
SERP_API_KEY = "e94af6c8514fa35bad847c9ba4c82193e015ad2f39a29d524560b986c0b3f365"
SERP_URL = "https://serpapi.com/search.json"

query = "Software Engineer Jobs"
num_pages = 3

all_jobs = []

for page in range(num_pages):

    params = {
        "engine": "google_jobs",
        "q": query,
        "api_key": SERP_API_KEY,
        "hl": "en",
        "start": page * 10
    }

    try:
        response = requests.get(
            SERP_URL,
            params=params,
            timeout=15
        )

        if response.status_code == 200:

            data = response.json()

            if "jobs_results" in data:

                for job in data["jobs_results"]:

                    all_jobs.append({
                        "Title": job.get("title"),
                        "Company": job.get("company_name"),
                        "Location": job.get("location"),
                        "Description": job.get("description"),
                        "Via": job.get("via")
                    })

            print(f"Page {page + 1} fetched successfully.")

        else:
            print(f"Error {response.status_code} on page {page + 1}")

    except Exception as e:
        print(f"Exception on page {page + 1}: {e}")

jobs_df = pd.DataFrame(all_jobs)

print(f"Total Jobs Collected: {len(jobs_df)}")

jobs_df.head()

Page 1 fetched successfully.
Page 2 fetched successfully.
Page 3 fetched successfully.
Total Jobs Collected: 30


,Title,Company,Location,Description,Via
0,Full Stack Software Engineer – Teradata/ETL/Da...,Ford Motor Company,"Dearborn, MI","At Ford Motor Company, we believe freedom of m...",Ford Careers
1,Cleared Software Engineer,Insight Global,"Sterling Heights, MI",Duration: 1 year contract to hire\n\nLocation:...,LinkedIn
2,Electrified Powertrain Controls Software Engineer,Ford Motor Company,"Dearborn, MI",We made history and now we work to transform t...,Ford Careers
3,Junior Software Engineer,"Eccalon, LLC","Detroit, MI",Job Description\n\n\n\n\nWe are seeking a Juni...,LinkedIn
4,Entry-Level Software Developer,Epic,"Warren, MI",Please note that this position is based on our...,LinkedIn


In [ ]:
jobs_df.to_excel("software_engineer_jobs.xlsx", index=False)

print("Excel file saved successfully!")

from google.colab import files
files.download("software_engineer_jobs.xlsx")

Excel file saved successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>